# KUL CV GA2 — 5-Fold CV Threshold Calibration (ConvNeXt-Small 320)

本 notebook 在 Kaggle T4 GPU 上运行，约 45 分钟完成。

**工作流程：**
1. 将 750 张训练图像分成 5 折（KFold, shuffle, seed=42）
2. 对每折单独训练 ConvNeXt-Small (S1+S2，不含 S3)
3. 收集每折的 OOF（out-of-fold）TTA 概率（每折 150 张）
4. 将 5 折的 OOF 概率合并（750 张），在全量 OOF 上搜索最优阈值
5. 用原始 `final_model.pth`（已有全量重训版本）+ 新阈值生成测试集预测

**所需 Kaggle Dataset 输入：**
- `kul-computer-vision-ga-2-2026`（训练/测试数据）
- `convnext-small-final`（包含 `final_model.pth`，用于最终测试推理）

**输出（`/kaggle/working/kfold_convnext_small_320/`）：**
- `fold{k}/checkpoints/best_model.pth` — 每折最优 checkpoint
- `fold{k}/metrics/oof_probabilities.csv` — 每折 OOF 概率
- `fold{k}/metrics/thresholds.npy` — 每折单独阈值（备用）
- `oof_probabilities_all.csv` — 750×20 全量 OOF 概率
- `best_thresholds_kfold.npy` — 全量 OOF 搜索阈值（主要结果）
- `best_thresholds_kfold_avg.npy` — 5 折阈值均值（备用）
- `ap_per_class_kfold.csv` — OOF per-class AP
- `submission_kfold_convnext_small_320.csv` — 1500 行最终提交

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import average_precision_score, f1_score
from tqdm.auto import tqdm
from PIL import Image
import gc

print('torch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    !nvidia-smi

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 配置
# ══════════════════════════════════════════════════════════════════════

DATA_DIR  = Path('/kaggle/input/kul-computer-vision-ga-2-2026')
FINAL_CKPT = Path('/kaggle/input/convnext-small-final/final_model.pth')  # 已有全量重训模型
OUT       = Path('/kaggle/working/kfold_convnext_small_320')
OUT.mkdir(parents=True, exist_ok=True)

# 训练超参数（与本地 convnext_small_320 一致）
BACKBONE   = 'convnext_small'
FEAT_DIM   = 768
IMG_SIZE   = 320
BATCH_SIZE = 8
EVAL_BS    = 16
N_FOLDS    = 5
SEED       = 42
WEIGHT_DECAY = 1e-4
GRID_STEP    = 0.01   # 阈值搜索步长（0.05→0.95，共 91 个候选）

# S1: 冻结骨干 5 epoch；S2: 全网络微调 20 epoch (早停 patience=None)
S1_EPOCHS, S1_LR = 5,  1e-3
S2_EPOCHS, S2_LR = 20, 1e-4

NUM_WORKERS = 2
USE_AMP     = True

LABELS = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor',
]

print('Output dir:', OUT)
print('Final model:', FINAL_CKPT, '| exists:', FINAL_CKPT.exists())

In [ ]:
# ── 数据工具 ──────────────────────────────────────────────────────────
_MEAN, _STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def train_tfm(sz=320):
    return T.Compose([
        T.Resize((sz, sz)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])

def val_tfm(sz=320):
    return T.Compose([T.Resize((sz, sz)), T.ToTensor(), T.Normalize(_MEAN, _STD)])

class VOCDataset(Dataset):
    def __init__(self, df, data_dir, split='train', transform=None):
        self.df, self.data_dir, self.split = df, Path(data_dir), split
        self.transform = transform or val_tfm()
        self.has_labels = all(c in df.columns for c in LABELS) and (df[LABELS].values != -1).any()
        self.indices = list(df.index)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        arr = np.load(self.data_dir / self.split / 'img' / f'{self.split}_{idx}.npy')
        img = self.transform(Image.fromarray(arr))
        if self.has_labels:
            return img, torch.FloatTensor(self.df.loc[idx, LABELS].values.astype(float))
        return img, idx

def rle_encode(arr):
    pixels = np.concatenate([[0], arr.flatten(), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(SEED)
pin = device.type == 'cuda'
print('Data utilities ready.')

In [ ]:
# ── 模型 ──────────────────────────────────────────────────────────────
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_neg = 1.0 - probs
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)
        log_p  = torch.log(probs.clamp(min=self.eps))
        log_np = torch.log(probs_neg.clamp(min=self.eps))
        loss = targets * log_p + (1 - targets) * log_np
        with torch.no_grad():
            weights = (
                targets * (1 - probs).pow(self.gamma_pos)
                + (1 - targets) * probs.pow(self.gamma_neg)
            )
        return -(loss * weights).mean()

def build_model(pretrained=True):
    w = models.ConvNeXt_Small_Weights.IMAGENET1K_V1 if pretrained else None
    base = models.convnext_small(weights=w)
    features = nn.Sequential(base.features, base.avgpool)
    head = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(FEAT_DIM, 512), nn.ReLU(inplace=True),
        nn.Dropout(0.2), nn.Linear(512, len(LABELS)),
    )
    for layer in head.modules():
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
    return features, head

def freeze_features(features): 
    for p in features.parameters(): p.requires_grad = False

def unfreeze_features(features):
    for p in features.parameters(): p.requires_grad = True

print('Model utilities ready.')

In [ ]:
# ── 训练工具 ─────────────────────────────────────────────────────────
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and pin)

def run_epoch(features, head, loader, criterion, optimizer, train):
    features.train(train); head.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False, desc='train' if train else 'val  '):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
                feats = features(imgs).flatten(1)
                logits = head(feats)
                loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total += loss.item() * len(imgs)
    return total / len(loader.dataset)

@torch.no_grad()
def collect_oof_probs(features, head, loader):
    features.eval(); head.eval()
    probs = []
    for imgs, _ in loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
            p1 = torch.sigmoid(head(features(imgs).flatten(1)))
            p2 = torch.sigmoid(head(features(imgs.flip(-1)).flatten(1)))
        probs.append(((p1 + p2) / 2).float().cpu().numpy())
    return np.vstack(probs)

print('Training utilities ready.')

In [ ]:
# ── 5-Fold 训练 ──────────────────────────────────────────────────────
full_df = pd.read_csv(DATA_DIR / 'train' / 'train_set.csv', index_col='Id')
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_splits = list(kf.split(full_df))

oof_probs_all   = np.zeros((len(full_df), len(LABELS)))
oof_labels_all  = full_df[LABELS].values.astype(int)
fold_thresholds = []

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)

for fold_k, (train_idx, val_idx) in enumerate(fold_splits):
    print(f'\n{"="*60}')
    print(f'Fold {fold_k+1}/{N_FOLDS}:  {len(train_idx)} train / {len(val_idx)} val')
    print('='*60)

    fold_dir  = OUT / f'fold{fold_k}'
    ckpt_dir  = fold_dir / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)
    met_dir   = fold_dir / 'metrics';     met_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = ckpt_dir / 'best_model.pth'

    set_seed(SEED + fold_k)
    train_df = full_df.iloc[train_idx]
    val_df   = full_df.iloc[val_idx]

    train_ds = VOCDataset(train_df, DATA_DIR, 'train', train_tfm(IMG_SIZE))
    val_ds   = VOCDataset(val_df,   DATA_DIR, 'train', val_tfm(IMG_SIZE))
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=pin)
    val_loader   = DataLoader(val_ds,   EVAL_BS,    shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

    features, head = build_model(pretrained=True)
    features, head = features.to(device), head.to(device)
    best_val_loss  = float('inf')
    history = []

    for stage, lr, epochs, frz in [('S1', S1_LR, S1_EPOCHS, True), ('S2', S2_LR, S2_EPOCHS, False)]:
        if frz:
            freeze_features(features)
            params = head.parameters()
            print(f'  [Fold {fold_k}][S1] head-only')
        else:
            unfreeze_features(features)
            params = list(features.parameters()) + list(head.parameters())
            print(f'  [Fold {fold_k}][S2] full fine-tune')

        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

        for epoch in range(1, epochs + 1):
            tl = run_epoch(features, head, train_loader, criterion, optimizer, True)
            vl = run_epoch(features, head, val_loader,   criterion, optimizer, False)
            scheduler.step()
            is_best = vl < best_val_loss
            if is_best:
                best_val_loss = vl
                torch.save({'features': features.state_dict(), 'head': head.state_dict()}, best_ckpt)
            history.append({'fold': fold_k, 'stage': stage, 'epoch': epoch,
                             'train_loss': tl, 'val_loss': vl, 'is_best': is_best})
            flag = '  <- best' if is_best else ''
            print(f'  [Fold {fold_k}][{stage}] Epoch {epoch:2d}/{epochs}  '
                  f'train={tl:.4f}  val={vl:.4f}{flag}')

    pd.DataFrame(history).to_csv(met_dir / 'training_history.csv', index=False)

    # Reload best and collect OOF
    ckpt = torch.load(best_ckpt, map_location=device)
    features.load_state_dict(ckpt['features'])
    head.load_state_dict(ckpt['head'])
    oof_probs = collect_oof_probs(features, head, val_loader)
    oof_probs_all[val_idx] = oof_probs

    oof_df = pd.DataFrame(oof_probs, columns=LABELS, index=val_df.index)
    oof_df.index.name = 'Id'
    oof_df.to_csv(met_dir / 'oof_probabilities.csv')
    print(f'  [Fold {fold_k}] OOF saved. Best val loss: {best_val_loss:.4f}')

    del features, head, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\nAll folds complete.')

In [ ]:
# ── 汇总 OOF 概率 + 校准阈值 ─────────────────────────────────────────
t_grid = np.arange(0.05, 0.96, GRID_STEP)

# Per-fold 阈值（备用）
for fold_k, (_, val_idx) in enumerate(fold_splits):
    fold_probs  = oof_probs_all[val_idx]
    fold_labels = oof_labels_all[val_idx]
    fold_t = []
    for c in range(len(LABELS)):
        best_t, best_f1 = 0.5, 0.0
        for t in t_grid:
            preds = (fold_probs[:, c] > t).astype(int)
            f1 = f1_score(fold_labels[:, c], preds, zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        fold_t.append(best_t)
    fold_t_arr = np.array(fold_t)
    fold_thresholds.append(fold_t_arr)
    np.save(OUT / f'fold{fold_k}' / 'metrics' / 'thresholds.npy', fold_t_arr)

# 全量 OOF 阈值搜索（主要结果）
ap_rows, pooled_thresholds = [], []
for c, label in enumerate(LABELS):
    best_t, best_f1 = 0.5, 0.0
    for t in t_grid:
        preds = (oof_probs_all[:, c] > t).astype(int)
        f1 = f1_score(oof_labels_all[:, c], preds, zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    pooled_thresholds.append(best_t)
    ap = average_precision_score(oof_labels_all[:, c], oof_probs_all[:, c])
    ap_rows.append({'class': label, 'ap': ap, 'best_threshold': best_t, 'best_f1': best_f1})

pooled_thresholds = np.array(pooled_thresholds)
avg_thresholds    = np.stack(fold_thresholds).mean(axis=0)

np.save(OUT / 'best_thresholds_kfold.npy',     pooled_thresholds)
np.save(OUT / 'best_thresholds_kfold_avg.npy', avg_thresholds)

ap_df      = pd.DataFrame(ap_rows)
map_score  = float(ap_df['ap'].mean())
ap_df.to_csv(OUT / 'ap_per_class_kfold.csv', index=False)

oof_full_df = pd.DataFrame(oof_probs_all, columns=LABELS, index=full_df.index)
oof_full_df.index.name = 'Id'
oof_full_df.to_csv(OUT / 'oof_probabilities_all.csv')

pd.DataFrame([{'experiment': 'convnext_small_320_kfold', 'n_folds': N_FOLDS,
               'oof_mAP': map_score}]).to_csv(OUT / 'kfold_summary.csv', index=False)

print(f'\n{"Class":<15s}  {"OOF AP":>6s}  {"Threshold":>9s}  {"OOF F1":>6s}')
print('-' * 47)
for _, row in ap_df.iterrows():
    print(f"{row['class']:<15s}  {row['ap']:6.4f}  {row['best_threshold']:9.3f}  {row['best_f1']:6.4f}")
print('-' * 47)
print(f'{"OOF mAP":<15s}  {map_score:6.4f}')

In [ ]:
# ── 测试集推理：用已有 final_model.pth + 新阈值 ──────────────────────
# 若 final_model.pth 不存在，回退到本折训练得到的 fold0/best_model.pth
if not FINAL_CKPT.exists():
    print(f'WARNING: {FINAL_CKPT} not found, falling back to fold0/best_model.pth')
    FINAL_CKPT = OUT / 'fold0' / 'checkpoints' / 'best_model.pth'

print(f'Loading model from {FINAL_CKPT}')
state = torch.load(FINAL_CKPT, map_location=device)

w = models.ConvNeXt_Small_Weights.IMAGENET1K_V1
base = models.convnext_small(weights=None)
features = nn.Sequential(base.features, base.avgpool).to(device)
head = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(FEAT_DIM, 512), nn.ReLU(inplace=True),
    nn.Dropout(0.2), nn.Linear(512, len(LABELS)),
).to(device)

# 支持两种 checkpoint 格式
if isinstance(state, dict) and 'features' in state and 'head' in state:
    features.load_state_dict(state['features'])
    head.load_state_dict(state['head'])
elif isinstance(state, dict) and any(k.startswith('features.') for k in state):
    # MultiLabelClassifier.state_dict() 格式（本地脚本保存）
    feat_state = {k[len('features.'):]: v for k, v in state.items() if k.startswith('features.')}
    head_state = {k[len('classifier.'):]: v for k, v in state.items() if k.startswith('classifier.')}
    features.load_state_dict(feat_state)
    head.load_state_dict(head_state)
else:
    raise ValueError('Unrecognized checkpoint format')

features.eval(); head.eval()

test_df = pd.read_csv(DATA_DIR / 'test' / 'test_set.csv', index_col='Id')
test_ds = VOCDataset(test_df, DATA_DIR, 'test', val_tfm(IMG_SIZE))
test_loader = DataLoader(test_ds, EVAL_BS, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=pin)

all_probs = []
with torch.no_grad():
    for imgs, _ in tqdm(test_loader, desc='Test TTA'):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP and pin):
            p1 = torch.sigmoid(head(features(imgs).flatten(1)))
            p2 = torch.sigmoid(head(features(imgs.flip(-1)).flatten(1)))
        all_probs.append(((p1 + p2) / 2).float().cpu().numpy())

test_probs = np.vstack(all_probs)
test_ids   = list(test_ds.indices)
print(f'Test probs shape: {test_probs.shape}')

In [ ]:
# ── 生成提交 CSV ─────────────────────────────────────────────────────
thresholds = pooled_thresholds   # 主要结果：全量 OOF 搜索
preds = (test_probs > thresholds[None, :]).astype(int)

prob_df = pd.DataFrame(test_probs, columns=LABELS, index=test_ids); prob_df.index.name = 'Id'
pred_df = pd.DataFrame(preds,      columns=LABELS, index=test_ids); pred_df.index.name = 'Id'
prob_df.to_csv(OUT / 'test_probabilities_kfold.csv')
pred_df.to_csv(OUT / 'test_binary_predictions_kfold.csv')

rows = {'Id': [], 'Predicted': []}
for idx in pred_df.index:
    rows['Id'].append(f'{idx}_classification')
    rows['Predicted'].append(rle_encode(pred_df.loc[idx, LABELS].values.astype(int)))
    rows['Id'].append(f'{idx}_segmentation')
    rows['Predicted'].append('')

sub_path = OUT / 'submission_kfold_convnext_small_320.csv'
sub = pd.DataFrame(rows).set_index('Id')
sub.to_csv(sub_path)

print(f'Submission: {sub_path}  ({len(sub)} rows)')
print('\n最终汇总：')
print(f'  OOF mAP:               {map_score:.4f}')
print(f'  Pooled threshold file: {OUT / "best_thresholds_kfold.npy"}')
print(f'  Avg threshold file:    {OUT / "best_thresholds_kfold_avg.npy"}')
print(f'  Submission:            {sub_path}')
sub.head(4)